# 🗐 Caixa - Extração de Dados

> Cerca de 80% dos dados financeiros e estratégicos da caixa são registrados em documentos no formato PDF. Com esse projeto, será possível encontrar todos os dados de maneira simplificada

### Para aprendizado

In [ ]:
### Por que esse código não detecta a linha de baixo?

# def reduzir_linha_produto(linha: str) -> str:
#     partes = linha.split()
    
#     numeros = [
#         p for p in partes
#         if re.search(r"\d", p)
#     ]
    
#     # 🔒 Proteção contra lista vazia
#     if len(numeros) < 2:
#         return linha  # não reduz
    
#     primeiro_valor = numeros[0]
#     ultimo_valor = numeros[-1]

#     for produto in PRODUTOS_ORDENADOS:
#         if linha.startswith(produto):
#             return f"{produto} {primeiro_valor} {ultimo_valor}"

#     return linha


# # regras de limpeza de dados
# def reduzir_texto(texto: str) -> str:
#     linhas = [l.strip() for l in texto.splitlines() if l.strip()]
#     saida = []

#     for linha in linhas:
        
#         # eu posso substituir todos os ifs de cada fórmula por uma condição dessas
#         if not any(linha.startswith(p) for p in PRODUTOS) and not linha.startswith("Estágio") and not linha.startswith("Total") and not linha.startswith("Individual") and not linha.startswith("Consolidado"):
#             continue
        
#         # faz um tratamento no estágio 1
#         if linha.startswith("Estágio 1 (1)"):
#             linha = linha.replace("Estágio 1 (1)", "Estágio 1")
        
#         # Redução dos produtos
#         if eh_produto(linha):
#             saida.append(reduzir_linha_produto(linha))
#         else:
#             saida.append(linha)
            
#     return "\n".join(saida)

## 󠀠⛏ Extração ( BRGaap )

> Dados de Exposição Bruta

> MELHORIAS # Só funciona para 3T25 pra frente. Antes disso precisa adaptar o código para só uma tabela

In [1]:
import pdfplumber
import re
import numpy as np
from datetime import datetime
import pandas as pd

31 a 36 para 3T25
48 a 53 para 2T25
(c) Composição por carteiras de crédito e faixas de vencimentos para 2T25
(c) Composição da carteira por faixa de atraso para 3T25

In [ ]:

with pdfplumber.open("BrGaap - Demonstrações Contábeis 1T25.pdf") as pdf:
    paginas = pdf.pages[47:53]  # intervalo desejado

    textos = []
    for page in paginas:
        txt = page.extract_text()
        if txt:
            textos.append(txt)

# Junta todas as páginas
text = "\n".join(textos)

# Pré-processamento
text = "\n".join(line.strip() for line in text.splitlines() if line.strip())

# Define início e fim
start = text.find("(b) Movimentação por estágios da carteira de crédito")
end = text.find("(c) Composição por carteiras de crédito e faixas de vencimentos", start)

trecho = text[start:end]

print(trecho)

linhas = [l.strip() for l in trecho.splitlines() if l.strip()]


(b) Movimentação por estágios da carteira de crédito
Individual
Constituição/
Transferência do/ Transferência do/ Saldo em
Estágio 1 (1) Saldo em 01/01/2025 (liquidação)/
para o estágio 2 para estágio 3 30/06/2025
variação
Empréstimos e direitos creditórios descontados
168.993.688 7.253.900 120.207 2.484.000 178.851.795
Financiamentos
7.910.356 437.678 218.399 142.215 8.708.648
Financiamentos rurais
57.442.005 (11.400.871) 3.057.381 1.421.073 50.519.588
Financiamentos imobiliários
786.474.231 28.621.785 4.553.604 2.653.397 822.303.017
Financiamentos de infraestrutura
100.829.120 1.353.457 329.313 10.163 102.522.053
Cessão de crédito
3.293.659 (315.990) 12.713 12.160 3.002.542
Outros créditos com características de concessão de crédito
16.078.584 (14.330.711) 8.518 139.374 1.895.765
Total 1.141.021.643 11.619.248 8.300.135 6.862.382 1.167.803.408
(1)Inclui o montante de R$ 42.045.228 referente aos contratos com mais de 30 dias de atraso.
Individual
Constituição/
Saldo em Transferência d

In [ ]:
PRODUTOS = [
    "Financiamentos",
    "Financiamentos rurais",
    "Financiamentos imobiliários",
    "Financiamentos de infraestrutura",
    "Cessão de crédito",
    "Outros ativos com características de concessão de crédito",
    "Total"
]

def eh_produto(linha: str) -> bool:
    
    return any(linha.startswith(p) for p in PRODUTOS)



# Código para evitar que o código só leia a primeira palavra de PRODUTOS
PRODUTOS_ORDENADOS = sorted(PRODUTOS, key=len, reverse=True)


def reduzir_texto(texto: str) -> str:
    linhas = [l.strip() for l in texto.splitlines() if l.strip()]
    saida = []

    i = 0
    while i < len(linhas):
        linha = linhas[i]

        # Mantém cabeçalhos importantes
        if linha.startswith(("Individual", "Consolidado", "Estágio")):
            if linha.startswith("Estágio 1 (1)"):
                linha = linha.replace("Estágio 1 (1)", "Estágio 1")
            saida.append(linha)
            i += 1
            continue

        # Detecta produto
        produto_encontrado = None
        for produto in PRODUTOS_ORDENADOS:
            if linha.startswith(produto):
                produto_encontrado = produto
                break

        if produto_encontrado:

            # 🔎 1️⃣ tenta extrair números da própria linha
            numeros = re.findall(r"[\(\)\d\.]+", linha)

            # Se não houver números suficientes, olha próxima linha
            if len(numeros) < 2 and i + 1 < len(linhas):
                proxima_linha = linhas[i + 1]
                numeros = re.findall(r"[\(\)\d\.]+", proxima_linha)
                i += 1  # pula linha de números

            if len(numeros) >= 2:
                primeiro_valor = numeros[0]
                ultimo_valor = numeros[-1]
                saida.append(f"{produto_encontrado} {primeiro_valor} {ultimo_valor}")
            else:
                # Se realmente não houver número, mantém só produto
                saida.append(produto_encontrado)

        i += 1

    return "\n".join(saida)


In [ ]:
texto = reduzir_texto(trecho)
print(texto)

Individual
Estágio 1 Saldo em 01/01/2025 (liquidação)/
Financiamentos 7.910.356 8.708.648
Financiamentos rurais 57.442.005 50.519.588
Financiamentos imobiliários 786.474.231 822.303.017
Financiamentos de infraestrutura 100.829.120 102.522.053
Cessão de crédito 3.293.659 3.002.542
Total 1.141.021.643 1.167.803.408
Individual
Estágio 2 (liquidação)/
Financiamentos 437.628 939.734
Financiamentos rurais 850.510 4.067.885
Financiamentos imobiliários 6.118.306 9.701.798
Financiamentos de infraestrutura 453.491 812.059
Cessão de crédito 26.741 31.262
Total 30.433.911 37.388.155
Individual
Estágio 3 (liquidação)/
Financiamentos 578.823 1.202.208
Financiamentos rurais 4.008.564 5.236.255
Financiamentos imobiliários 32.603.630 34.725.115
Financiamentos de infraestrutura 5.865.448 5.765.645
Cessão de crédito 76.597 88.601
Total 65.401.947 72.809.524
Consolidado
Estágio 1 (liquidação)/
Financiamentos 7.910.356 8.708.648
Financiamentos rurais 57.442.005 50.519.588
Financiamentos imobiliários 786.47

### Extração de datas

In [ ]:
# Extração de anos e meses
# para criar a coluna de trimestre

from datetime import datetime

padrao_data = re.compile(r"\b\d{2}/\d{2}/\d{4}\b")
datas = padrao_data.findall(trecho)

print(datas)

def trimestre_from_date(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    trimestre_map = {1: "1T", 3: "1T", 6: "2T", 9: "3T", 12: "4T"}
    trimestre = trimestre_map.get(dt.month)

    if not trimestre:
        raise ValueError(f"Mês inesperado na data: {date_str}")

    return f"{trimestre}{str(dt.year)[-2:]}"

def extract_anos(texto: str) -> list[str]:
    datas = re.findall(r"\d{2}/\d{2}/\d{4}", texto)

    if len(datas) < 2:
        raise ValueError("Não foi possível encontrar duas datas finais")
    
    datas_finais = datas[-2:]
    return [trimestre_from_date(d) for d in datas_finais]
    

['01/01/2025', '30/06/2025', '01/01/2025', '30/06/2025', '01/01/2025', '30/06/2025', '01/01/2025', '30/06/2025', '01/01/2025', '30/06/2025', '01/01/2025', '30/06/2025']


In [ ]:
extract_anos(trecho)

['1T25', '2T25']

### Individual x Consolidado
É o que vai definir se o valor que será extraído será com as células dentro de "Individual" ou "Consolidado"

In [ ]:
def categorias(texto: str, categoria_escolhida: str):
    resultado = []
    categoria_atual = None

    linhas = [l.strip() for l in texto.splitlines() if l.strip()]

    for linha in linhas:

        # Detecta categoria (case insensitive)
        if re.fullmatch(r"Individual", linha, flags=re.IGNORECASE):
            categoria_atual = "Individual"
            continue

        elif re.fullmatch(r"Consolidado", linha, flags=re.IGNORECASE):
            categoria_atual = "Consolidado"
            continue

        # Só captura se estiver dentro da categoria escolhida
        if categoria_atual == categoria_escolhida:
            resultado.append(linha)

    return resultado


print("\n".join(categorias(texto, "Individual")))



Estágio 1 Saldo em 01/01/2025 (liquidação)/
Financiamentos 7.910.356 8.708.648
Financiamentos rurais 57.442.005 50.519.588
Financiamentos imobiliários 786.474.231 822.303.017
Financiamentos de infraestrutura 100.829.120 102.522.053
Cessão de crédito 3.293.659 3.002.542
Total 1.141.021.643 1.167.803.408
Estágio 2 (liquidação)/
Financiamentos 437.628 939.734
Financiamentos rurais 850.510 4.067.885
Financiamentos imobiliários 6.118.306 9.701.798
Financiamentos de infraestrutura 453.491 812.059
Cessão de crédito 26.741 31.262
Total 30.433.911 37.388.155
Estágio 3 (liquidação)/
Financiamentos 578.823 1.202.208
Financiamentos rurais 4.008.564 5.236.255
Financiamentos imobiliários 32.603.630 34.725.115
Financiamentos de infraestrutura 5.865.448 5.765.645
Cessão de crédito 76.597 88.601
Total 65.401.947 72.809.524


### Blocagem de estágios

In [ ]:

def split_blocos_estagio(texto: str) -> dict:
    blocos = {}
    estagio_atual = None
    buffer = []

    for linha in texto.splitlines():
        
        # 🔥 Remove tudo após "Estágio X"
        linha = re.sub(
            r"(Estágio\s+\d+).*",
            r"\1",
            linha,
            flags=re.IGNORECASE
        ).strip()

        match = re.search(r"\bEstágio\s+(\d+)", linha, flags=re.IGNORECASE)

        if match:
            if estagio_atual is not None:
                blocos[estagio_atual] = "\n".join(buffer)
                buffer = []

            estagio_atual = int(match.group(1))
            continue

        if estagio_atual is not None:
            buffer.append(linha)

    if estagio_atual is not None:
        blocos[estagio_atual] = "\n".join(buffer)

    return blocos


In [ ]:
def parse_linha_produto(linha: str):
    for produto in sorted(PRODUTOS, key=len, reverse=True):
        if linha.startswith(produto):
            
            valores = linha[len(produto):].strip().split()
            
            return produto, valores

    return None, []


### Tratamento de formato do número

In [ ]:
def normalizar_valor(valor: str):
    if valor == "—":
        return None

    # remove espaços
    valor = valor.strip()

    # caso (123) → -123
    if re.fullmatch(r"\(\d+\)", valor):
        valor = "-" + valor[1:-1]

    # remove separador de milhar
    valor = valor.replace(".", "")

    return int(float(valor))

### Criação de um Dataframe

In [ ]:
def construir_linhas_caixa(texto_reduzido: str, trecho_original: str) -> list[dict]:
    
    anos = extract_anos(trecho_original)
    blocos = split_blocos_estagio(texto_reduzido)

    linhas_finais = []

    for estagio, bloco in blocos.items():
        
        for linha in bloco.splitlines():
            
            if eh_produto(linha):
                
                produto, valores = parse_linha_produto(linha)

                for idx, ano in enumerate(anos):
                    
                    valor = (
                        normalizar_valor(valores[idx])
                        if idx < len(valores)
                        else None
                    )

                    linhas_finais.append({
                        "ano": ano,
                        "banco": "Caixa",
                        "produto": produto,
                        "PD": "-",
                        "Estágio": estagio,
                        "Exposição Bruta": valor,
                    })

    return linhas_finais


In [ ]:
# como mudar o nome da última coluna para Exposição Bruta ou Perda Esperada?

# Padroniza os nomes
MAP_RENOMEAR = {
    "Financiamentos": "Financiamentos",
    "Financiamentos rurais": "Rural",
    "Financiamentos imobiliários": "Imobiliário",
    "Financiamentos de infraestrutura": "Infraestrutura",
    "Cessão de crédito": "Cessão de crédito",
    "Outros ativos com características de concessão de crédito": "Outros ativos",
}

# gera a tabela
texto_reduzido = reduzir_texto(trecho)
texto_individual = "\n".join(categorias(texto_reduzido, "Individual"))

linhas = construir_linhas_caixa(texto_individual, trecho)

df_final = pd.DataFrame(linhas)

# remove total
df_final = df_final.copy().replace(MAP_RENOMEAR)
df_final



,ano,banco,produto,PD,Estágio,Exposição Bruta
0,1T25,Caixa,Financiamentos,-,1,7910356
1,2T25,Caixa,Financiamentos,-,1,8708648
2,1T25,Caixa,Rural,-,1,57442005
3,2T25,Caixa,Rural,-,1,50519588
4,1T25,Caixa,Imobiliário,-,1,786474231
5,2T25,Caixa,Imobiliário,-,1,822303017
6,1T25,Caixa,Infraestrutura,-,1,100829120
7,2T25,Caixa,Infraestrutura,-,1,102522053
8,1T25,Caixa,Cessão de crédito,-,1,3293659
9,2T25,Caixa,Cessão de crédito,-,1,3002542


### 📊 Passar para o Excel

In [ ]:
import openpyxl
from openpyxl import load_workbook

In [ ]:

# Carrega arquivos
dados_bancarios_xlsx = load_workbook("Excel - Dados bancários.xlsx")
caixa_sheet = dados_bancarios_xlsx["Caixa"]

# Verifica se existe todos os anos ( se não, adiciona +1 coluna a direita )
anos_df = df_final["ano"].unique()

for ano_df in anos_df:
    
    existe = False
    
    for col in range(1, caixa_sheet.max_column + 1):
        if str(caixa_sheet.cell(row=5, column=col).value).strip() == str(ano_df):
            existe = True
            break
    
    if not existe:
        nova_coluna = caixa_sheet.max_column + 1
        caixa_sheet.cell(row=5, column=nova_coluna, value=ano_df)

# Identificação da coluna Tipo
ultima_coluna_df = df_final.columns[-1]

if ultima_coluna_df not in ["Exposição Bruta", "Perda Esperada"]:
    raise ValueError("Última coluna do DataFrame não é métrica válida.")

# Preenchimento dos dados
for _, row in df_final.iterrows():
    
    tipo_df = ultima_coluna_df
    estagio_df = f"Estágio {row['Estágio']}"
    produto_df = row["produto"]
    ano_df = row["ano"]
    valor_df = row[ultima_coluna_df]
    
    linha_encontrada = None
    coluna_encontrada = None
    
    # 🔎 Encontrar linha correta (Tipo + Estágio + Produto)
    for r in range(1, caixa_sheet.max_row + 1):
        
        tipo_excel = str(caixa_sheet.cell(row=r, column=2).value).strip()
        estagio_excel = str(caixa_sheet.cell(row=r, column=3).value).strip()
        produto_excel = str(caixa_sheet.cell(row=r, column=4).value).strip()
        
        if (
            tipo_excel == tipo_df and
            estagio_excel == estagio_df and
            produto_excel == produto_df
        ):
            linha_encontrada = r
            break
    
    # 🔎 Encontrar coluna correta (Ano na linha 5)
    for c in range(1, caixa_sheet.max_column + 1):
        if str(caixa_sheet.cell(row=5, column=c).value).strip() == str(ano_df):
            coluna_encontrada = c
            break
    
    # 🔥 Preencher célula
    if linha_encontrada and coluna_encontrada:
        caixa_sheet.cell(
            row=linha_encontrada,
            column=coluna_encontrada,
            value=valor_df
        )

# Salva arquivo
dados_bancarios_xlsx.save("Excel - Dados bancários.xlsx")


## 󠀠⛏ Extração ( IFRS )

> Dados de Perdas Esperadas

In [20]:
# Ingerir PDF
import pdfplumber
import pandas as pd
import re
import numpy as np
import re

In [21]:
with pdfplumber.open("Demonstrações contábeis consolidadas Caixa 2T25.pdf") as pdf:
    page = pdf.pages[51]   # página 53 (índice começa em 0)
    text = page.extract_text()


# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("(a) Movimentação da provisão para perdas esperadas")
end = text.find("(b) Movimentação da provisão para perdas esperadas", start)

trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()]

(a) Movimentação da provisão para perdas esperadas
Saldo em Constituição/ Transferência do/ para Transferência do/ para Saldo em
Estágio 1
31/12/2024 (reversão) estágio 2 estágio 3 30/06/2025
Empréstimos e direitos creditórios descontados (8.366.631) 1.183.160 1.939.179 1.280.197 (3.964.095)
Financiamentos (263.576) 13.504 59.675 48.588 (141.809)
Financiamentos rurais e agroindustriais (1.414.489) 340.925 256.982 214.370 (602.212)
Financiamentos imobiliários (6.702.364) 1.918.808 (307.730) 286.260 (4.805.026)
Financiamentos de infraestrutura (625.891) (30.945) (408) 23 (657.221)
Outros ativos (849.238) 64.789 252.725 77.253 (454.471)
Total (18.222.189) 3.490.241 2.200.423 1.906.691 (10.624.834)
Saldo em Constituição/ Transferência do/ para Transferência do/para Saldo em
Estágio 1
31/12/2023 (reversão) estágio 2 estágio 3 31/12/2024
Empréstimos e direitos creditórios descontados (7.890.186) (560.072) (153.942) 237.569 (8.366.631)
Financiamentos (113.304) (154.655) (2.933) 7.316 (263.576

In [22]:
# Qual será o formato da tabela, neste caso?

## 󠀠⛏ Extração de dados - ( Relatório de Desempenho )

> Dados relacionados a Qt Clientes, Receita do Cartão e Despesas de Provisão são encontradas aqui

In [2]:
import pdfplumber
import pandas as pd
import re
import numpy as np
from datetime import datetime

# bibliotecas de visão computacional
import pytesseract

In [ ]:
import camelot

tables = camelot.read_pdf(
    "Relatório de Análise de Desempenho 3T25 - Caixa.pdf",
    page = pdf.pages[6]
    flavor="stream"
)

df = tables[0].df
print(df)

                                           0       1       2       3       4  \
0                  Índice de Capital Nível I   15,05   14,53    0,52   14,20   
1  Indicadores da Carteira de Crédito (em %)    3T25    2T25  Δ p.p.    3T24   
2    Inadimplência Total (atrasos > 90 dias)    3,01    2,66    0,35    2,27   
3                     Livres Pessoas Físicas    5,68    5,57    0,11    4,46   
4                   Livres Pessoas Jurídicas   12,22   11,02    1,20    7,76   
5                               Imobiliário6    1,30    1,26    0,04    1,42   
6                             Infraestrutura    0,04    0,01    0,02    0,52   
7                                Agronegócio   11,20    7,02    4,19    3,35   
8                               PCLD/Crédito    4,44    4,24    0,20    4,09   
9                       Cobertura > 90 dias7  148,09  163,76  -15,67  180,30   

        5       6       7       8  
0    0,85   15,05   14,20    0,85  
1  Δ p.p.    9M25    9M24  Δ p.p.  
2    0,74  

Quais os nomes dos indicadores principais?
ROAE Recorrente
Qtd Clientes